In [4]:
!pip uninstall -y sparkmagic

In [5]:
!pip install -r require_phase1_20250316.txt -q

In [9]:
import os
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Uncomment to suppress TensorFlow warnings if needed

import faiss
import numpy as np
import logging
import ipywidgets as widgets
import boto3
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from typing import List
from IPython.display import display
from functools import lru_cache
from sentence_transformers import SentenceTransformer
import unicodedata
import re
import traceback
import chardet  # Import chardet

# Setup logging
logging.basicConfig(level=logging.INFO)  # Set to DEBUG for more verbose logging
logger = logging.getLogger(__name__)

# Configuration Dictionary (Only Selected Models)
CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "json_folder": "dev/json",
    "index_folder": "dev/idx",
    "nlist": 512,
    "nprobe": 16,
    "num_training_samples": 1000,
    "embedding_models": {
        "amazon.titan-embed-text-v2:0": 1536,
        "anthropic.claude-3-sonnet-20240229-v1:0": 8192,
        "meta.llama3-70b-instruct-v1:0": 8192
    },
    "default_model": "amazon.titan-embed-text-v2:0"
}

# Initialize S3 and Bedrock clients
s3_client = boto3.client("s3")
bedrock = boto3.client("bedrock-runtime")
query_cache = {}
faiss_lock = Lock()  # Lock for FAISS index updates


# This function requires ObjectTagging Read/Write permissions
def check_s3_object_tag(bucket: str, key: str, tag_key: str) -> bool:
    """Checks if an object in S3 has a specific tag."""
    try:
        response = s3_client.get_object_tagging(Bucket=bucket, Key=key)
        tags = {tag["Key"]: tag["Value"] for tag in response.get("TagSet", [])}
        return tag_key in tags
    except s3_client.exceptions.ClientError as e:
        if e.response['Error']['Code'] == 'AccessDenied':
            logging.warning(f"Skipping tag check for {key} due to insufficient permissions.")
            return False  # Assume the object has no tag and continue
        else:
            logging.error(f"Error retrieving tags for {key}: {e}")
            return False


def list_objects_s3(bucket: str, prefix: str) -> List[str]:
    """Lists all objects in an S3 bucket under a given prefix."""
    try:
        response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
        return [obj["Key"] for obj in response.get("Contents", [])]
    except Exception as e:
        logging.error(f"⚠️ Error listing S3 objects: {e}")
        return []


# Load FAISS index from S3
def load_faiss_index(model_name):
    bucket = CONFIG["bucket_name"]
    index_key = f"{CONFIG['index_folder']}/{model_name}.faiss"  # Include model name in index key

    try:
        response = s3_client.get_object(Bucket=bucket, Key=index_key)
        index_data = np.frombuffer(response["Body"].read(), dtype=np.uint8)
        index = faiss.deserialize_index(index_data)
        logger.info(f"✅ Loaded FAISS index for {model_name} from S3.")
    except s3_client.exceptions.NoSuchKey:
        logger.info(
            f"⚠️ No existing FAISS index found for {model_name}, creating a new one. This may take some time...")
        index = faiss.IndexFlatL2(CONFIG["embedding_models"][model_name])
    except Exception as e:
        logger.error(f"❌ Error loading FAISS index for {model_name}: {e}. Creating a new one.")
        index = faiss.IndexFlatL2(CONFIG["embedding_models"][model_name])

    return index, index_key


# Save FAISS index to S3
def save_faiss_index(index, index_key):
    with faiss_lock:  # Prevent multiple threads from writing at the same time
        try:
            index_data = faiss.serialize_index(index).tobytes()
            s3_client.put_object(Bucket=CONFIG["bucket_name"], Key=index_key, Body=index_data)
            logger.info(f"✅ FAISS index saved to {index_key}")
        except Exception as e:
            logger.error(f"❌ Failed to save FAISS index to {index_key}: {e}\n{traceback.format_exc()}")


# Clean text function to remove unwanted characters and normalize spaces
def clean_text(text: str) -> str:
    """
    Cleans extracted text by normalizing Unicode characters, removing extra newlines,
    and collapsing multiple whitespace characters.
    """
    text = unicodedata.normalize("NFKC", text)
    # Remove multiple newlines but keep single newline for paragraph breaks
    text = re.sub(r'\n{2,}', '\n', text)
    # Replace newline and tab characters with a space
    text = text.replace('\n', ' ').replace('\t', ' ')
    # Collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Improved s3_read_json function with encoding detection
def read_s3_json(bucket, key):
    try:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        content = response["Body"].read()

        # Attempt to detect encoding
        encoding = chardet.detect(content)['encoding']
        logger.debug(f"Detected encoding for {key}: {encoding}")

        # Decode with detected encoding, or fall back to utf-8
        try:
            file_content_str = content.decode(encoding or 'utf-8')
        except UnicodeDecodeError:
            logger.warning(f"Decoding with {encoding} failed, trying utf-8 with errors='ignore' for {key}")
            file_content_str = content.decode('utf-8', errors='ignore')

        logger.debug(f"File content (first 100 chars): {file_content_str[:100]}...")  # Debug: Print first 100 chars
        return json.loads(file_content_str)

    except json.JSONDecodeError as e:
        logger.error(f"Invalid JSON format in {key}: {e}\n{traceback.format_exc()}")
        return None
    except Exception as e:
        logger.error(f"Failed to load {key}: {e}\n{traceback.format_exc()}")
        return None


# Bedrock Embedding Function with Cache and document-level embedding

logger = logging.getLogger(__name__)

def generate_embedding(text, model_name, max_retries=1):
    """
    Generates an embedding using AWS Bedrock.
    - Checks the cache to avoid redundant API calls.
    - Handles missing or malformed responses properly.
    - Implements a retry mechanism for transient failures.
    """
    cache_key = (text, model_name)
    
    if cache_key in query_cache:
        logger.info(f"⚡ Using cached embedding for '{text}' ({model_name})")
        return query_cache[cache_key]

    for attempt in range(max_retries + 1):
        try:
            logger.info(f"⚡ Requesting embedding from AWS Bedrock for {model_name} (Attempt {attempt + 1})...")
            start_time = time.time()

            # ✅ Make the request
            response = bedrock.invoke_model(
                modelId=model_name,
                contentType="application/json",
                accept="application/json",
                body=json.dumps({"inputText": text})
            )

            elapsed = time.time() - start_time
            logger.info(f"🕒 Bedrock embedding request took {elapsed:.2f} seconds.")

            # ✅ Log only metadata (NOT full response) unless an issue arises
            logger.info(f"📌 AWS Bedrock Response Metadata: {response.get('ResponseMetadata', {})}")

            # ✅ Validate response structure
            if not response:
                logger.error(f"❌ AWS Bedrock response is empty for {model_name}. Retrying...")
                continue

            if response.get("ResponseMetadata", {}).get("HTTPStatusCode") != 200:
                logger.error(f"❌ AWS Bedrock returned HTTP {response['ResponseMetadata']['HTTPStatusCode']} for {model_name}. Retrying...")
                continue

            if "Body" not in response:
                logger.error(f"❌ AWS Bedrock response for {model_name} is missing 'Body'. Retrying...")
                continue

            # ✅ Read the response body safely
            response_body_bytes = response["Body"].read()  # <--- StreamingBody needs to be read
            if not response_body_bytes:
                logger.error(f"❌ AWS Bedrock response for {model_name} has an empty 'Body'. Retrying...")
                continue

            # ✅ Decode JSON correctly
            try:
                response_body = json.loads(response_body_bytes.decode("utf-8"))
            except json.JSONDecodeError as e:
                logger.error(f"❌ JSON decoding error for {model_name}: {e}. Retrying...")
                continue

            # ✅ Ensure 'embedding' exists in response
            if "embedding" not in response_body:
                logger.error(f"❌ 'embedding' key missing in Bedrock response for {model_name}. Full response: {response_body}")
                return []

            embedding = response_body["embedding"]

            # ✅ Ensure embedding dimension matches expected value
            expected_dim = CONFIG["embedding_models"].get(model_name, None)
            if expected_dim and len(embedding) != expected_dim:
                logger.warning(f"⚠️ Embedding dimension mismatch for {model_name}: Expected {expected_dim}, got {len(embedding)}")
                return []

            # ✅ Store in cache
            query_cache[cache_key] = embedding
            return embedding

        except boto3.exceptions.Boto3Error as e:
            logger.error(f"❌ AWS SDK error for {model_name}: {e}. Retrying...")
        except Exception as e:
            logger.error(f"❌ Unexpected error generating embedding for {model_name}: {e}. Retrying...")

    logger.error(f"❌ Embedding generation failed for {model_name} after {max_retries + 1} attempts.")
    return []


def process_single_file(obj, model_name):
    """
    Processes a single JSON file from S3:
    - Reads the file.
    - Combines text from all chunks to create a single document-level text.
    - Generates an embedding for the combined text.
    - Updates the JSON file with the new embedding.
    """
    bucket = CONFIG["bucket_name"]
    file_key = obj["Key"]

    logger.info(f"Processing file: {file_key}")  # Log processing

    file_content = read_s3_json(bucket, file_key)  # Reads the JSON file
    if file_content is None:
        return  # Stop processing file due to failure

    # ✅ Combine all text from chunks into one document-level string
    combined_text = " ".join(clean_text(chunk.get("text", "").strip()) for chunk in file_content.get("chunks", []))

    logger.debug(f"Combined text (first 100 chars): {combined_text[:100]}...")  # Debugging

    if combined_text:
        # ✅ Ensure the 'embeddings' field exists
        if "embeddings" not in file_content:
            file_content["embeddings"] = {}

        # ✅ Generate embedding
        embedding_result = generate_embedding(combined_text, model_name)
        if not embedding_result:
            logger.error(f"❌ No embedding generated for {file_key}. Skipping write.")
            return

        # ✅ Store the embedding under the correct model name
        file_content["embeddings"][model_name] = embedding_result

        try:
            # ✅ Save back to S3
            json_body = json.dumps(file_content, ensure_ascii=False).encode("utf-8")
            s3_client.put_object(Bucket=bucket, Key=file_key, Body=json_body)
            logger.info(f"✅ Embeddings successfully saved in {file_key}")

            # ✅ Ensure embedding was correctly written
            if "embeddings" not in file_content or model_name not in file_content["embeddings"]:
                logger.error(f"❌ Embeddings were NOT saved in {file_key}. Check AWS Bedrock response.")
            else:
                logger.info(f"✅ Updated embeddings in {file_key}")

        except Exception as e:
            logger.error(f"❌ Failed to save {file_key}: {e}")
    else:
        logger.warning(f"⚠️ No text found to embed in {file_key}. Skipping.")


def run_process_in_background(model_name):
    """Starts processing of JSON files in a background thread."""
    thread = threading.Thread(target=process_json_files, args=(model_name,))
    thread.start()


def process_json_files(model_name):
    """
    Processes all JSON files in the configured S3 folder:
    - Loads or creates a FAISS index.
    - Retrieves a list of JSON files.
    - Processes each file in parallel to generate document-level embeddings.
    - Saves the FAISS index back to S3.
    """
    start_time = time.time()
    logger.info(f"🚀 Starting processing for model {model_name}...")
    bucket = CONFIG["bucket_name"]
    folder = CONFIG["json_folder"]
    index, index_key = load_faiss_index(model_name)

    logger.info("🔍 Retrieving file list from S3...")
    start_list_time = time.time()
    try:
        response = s3_client.list_objects_v2(Bucket=bucket, Prefix=folder)
    except Exception as e:
        logger.error(f"❌ Error listing objects from S3: {e}\n{traceback.format_exc()}")
        return

    logger.info(f"📌 S3 File List Retrieved in {time.time() - start_list_time:.2f} sec")
    if "Contents" not in response or len(response["Contents"]) == 0:
        logger.error(
            "❌ No JSON files found in S3 bucket. Ensure the folder is populated. No processing was performed.")
        return

    with ThreadPoolExecutor(max_workers=5) as executor:
        executor.map(process_single_file, response["Contents"], [model_name] * len(response["Contents"]))

    save_faiss_index(index, index_key)
    duration = time.time() - start_time
    logger.info(
        f"Finished processing all JSON files in {duration:.2f} seconds. Total files processed: {len(response['Contents'])}")


# UI Setup: Create widgets for model selection and execution.
model_selector = widgets.Dropdown(
    options=list(CONFIG["embedding_models"].keys()),
    value=CONFIG["default_model"],
    description="Embedding Model:",
)
execute_button = widgets.Button(
    description="Execute",
    button_style="success"
)


# Do not trigger processing on model change; only run when Execute is clicked.
execute_button.on_click(lambda _: run_process_in_background(model_selector.value))

# Display the UI immediately
display(model_selector, execute_button)

# ==============================================
# END OF SCRIPT
# ==============================================

Dropdown(description='Embedding Model:', options=('amazon.titan-embed-text-v2:0', 'anthropic.claude-3-sonnet-2…

Button(button_style='success', description='Execute', style=ButtonStyle())

INFO:__main__:🚀 Starting processing for model amazon.titan-embed-text-v2:0...
INFO:__main__:✅ Loaded FAISS index for amazon.titan-embed-text-v2:0 from S3.
INFO:__main__:🔍 Retrieving file list from S3...
INFO:__main__:📌 S3 File List Retrieved in 0.08 sec
INFO:__main__:Processing file: dev/json/
INFO:__main__:Processing file: dev/json/1806.00663_page_1.json
INFO:__main__:Processing file: dev/json/1806.00663_page_10.json
INFO:__main__:Processing file: dev/json/1806.00663_page_11.json
INFO:__main__:Processing file: dev/json/1806.00663_page_12.json
ERROR:__main__:Invalid JSON format in dev/json/: Expecting value: line 1 column 1 (char 0)
Traceback (most recent call last):
  File "/tmp/ipykernel_1601/312986812.py", line 142, in read_s3_json
    return json.loads(file_content_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/pyth